In [1]:
import polars as pl

from libs.DataLoader import Loader
from libs.RNNModel import RNNModel
from libs.Solver import Solver
from libs.Trainer import Trainer
from libs.WeightedAvgModel import WeightedAvgModel
from libs.constants import *
from libs.utils import NDCG, count_polars

In [2]:
EMB_DIM = 64
loader = Loader('ur0.01_ir0.01', content_embedding_size=EMB_DIM, batch_size=500_000)
(train_df, val_df), users_df, items_df = loader.load_data(convert_to_pandas=False)

Load metadata
Create lazy interaction datasets
Get unique users/items


  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Filter embeddings
Process metadata
Compute aggregates


  0%|          | 0/8 [00:00<?, ?it/s]

Filter interactions
Finalize interactions
Count
Train: 62_578 Val: 12_099 Users: 2_309 Items: 120_415


In [3]:
train_df: pl.LazyFrame = train_df.filter(pl.col(TARGET) > 0)  # Interaction: positive
test_df: pl.LazyFrame = val_df.filter(pl.col(TARGET) > 0)  # Interaction: positive

test = test_df.select(ITEM).unique(ITEM)
cold_test = test_df.join(train_df.select(ITEM).unique(), on=ITEM, how="anti")

print(f"Train: {count_polars(train_df):_} Test: {count_polars(test_df):_}")
print(f"Test items: {count_polars(test)} Test cold items: {count_polars(cold_test)}")

Train: 62_457 Test: 1_970
Test items: 1090 Test cold items: 405


In [4]:
from typing import Literal, Union
from numpy.typing import NDArray

import numpy as np
import polars as pl
import torch
from torch.nn.utils.rnn import pad_sequence
from torch import Tensor

from libs.BaseRecurrentModel import BaseRecurrentModel
from libs.constants import *
from libs.utils import build_embedding_sequences


class WeightedAvgModel(BaseRecurrentModel):
    """
    :param method: 'add' or 'mul'
    :param temporal_distribution: distribution of weights applied to temporal indices
    :param alpha: for 'add' method - weight of temporal component
    :param temp: for 'exp' temporal_distribution - multiplier in exp
    """

    def __init__(self, method: Literal['add', 'mul'] = 'add',
                 temporal_distribution: Literal['linear', 'exp', 'uni'] = 'linear',
                 alpha: float = 0.5, temp: float = 0.1):
        super().__init__(trainable=False)
        self.method = method
        self.temporal_distribution = temporal_distribution
        self.alpha = alpha
        self.temp = temp

    def process_data_batch(self, data_batch: pl.DataFrame, items_df: pl.LazyFrame,
                           mode: Literal['train', 'val', 'predict'] = 'train') -> Union[NDArray, Tensor]:
        device = next(self.parameters()).device if any(p.numel() for p in self.parameters()) else torch.device('cpu')

        item_lists = data_batch[ITEM].to_list()
        time_index_lists = data_batch[TIME_INDEX].to_list()
        target_lists = data_batch[TARGET].to_list()

        input = build_embedding_sequences(items_df, item_lists, device)  # (L, B, D)

        weights = pad_sequence([  # (L, B)
            torch.from_numpy(self._get_weights(row_time, row_target))
            for row_time, row_target in zip(time_index_lists, target_lists)
        ], batch_first=False, padding_value=0.0).to(dtype=torch.float32, device=device)

        out = (weights.unsqueeze(-1) * input) # (L, B, D)

        if mode in ['train', 'val']:
            return torch.stack([
                out[:t].sum(dim=0)
                for t in range(1, out.shape[0]+1)
            ])
        elif mode == 'predict':
            return out.sum(dim=0).detach().cpu().numpy().astype(np.float32)
        else:
            raise NotImplementedError()

    def _get_weights(self, indices: NDArray, ranks: NDArray) -> NDArray:
        indices = np.asarray(indices)
        ranks = np.asarray(ranks, dtype=float)
        if indices.shape != ranks.shape:
            raise NotImplementedError()

        n = len(indices)
        order = np.argsort(np.argsort(indices)) + 1

        if self.temporal_distribution == 'linear':
            temporal_w = order / order.sum()
        elif self.temporal_distribution == 'exp':
            exp = np.exp(self.temp * order)
            temporal_w = exp / np.sum(exp)
        elif self.temporal_distribution == 'uni':
            temporal_w = np.ones(n) / n
        else:
            raise NotImplementedError()

        mag_sum = ranks.sum()
        if mag_sum == 0:
            magnitude_w = np.ones_like(ranks) / n
        else:
            magnitude_w = ranks / mag_sum

        if self.method == 'mul':
            combined = temporal_w * magnitude_w
        elif self.method == 'add':
            combined = self.alpha * temporal_w + (1.0 - self.alpha) * magnitude_w
        else:
            raise NotImplementedError()

        return combined / combined.sum()

In [71]:
trainer = Trainer(
    # model=RNNModel(EMB_DIM, hidden_dim=32, num_layers=1, dropout=0.0),
    model=WeightedAvgModel('add', 'linear', alpha=0.5),
    train_interactions=train_df,
    predict_items=test,
    items_metadata=items_df,
    val_ratio=0.3,
    loss_type='cos',
    num_recent_videos=100,
)
solver = Solver(
    trainer=trainer,
    candidates_to_keep=500,
    top_per_item=100,
    max_per_user=100,
)

Temporal train/val split with ratio val_ratio 0.3
Count
All: 2309 Train: 2302 Val: 2203 Test: 1090


In [72]:
trainer.fit(epochs=10, users_batch_size=1000, verbose=False)

Model is not trainable


In [73]:
solver.collect_candidates(users_batch_size=1000, items_batch_size=1000)

data:   0%|          | 0/3 [00:00<?, ?it/s]

predict:   0%|          | 0/2 [00:00<?, ?it/s]

In [74]:
results = solver.solve()

predict = pl.DataFrame({
    ITEM: list(results.keys()),
    USER: [[u for u, _ in users_scores] for users_scores in results.values()]
})

In [75]:
for pred, pred_name in [
    (predict, "constrained")
]:
    for true, true_name in [
        (test_df, "all"),
        (test_df.join(cold_test, on=ITEM, how='inner'), "cold")
    ]:
        pl.LazyFrame().group_by().agg()
        print(f"Predict: {pred_name} Test: {true_name} NDCG: {NDCG(pred.to_pandas(), true.group_by(ITEM).agg(pl.col(USER)).collect().to_pandas()):.4f}")

Predict: constrained Test: all NDCG: 0.0942
Predict: constrained Test: cold NDCG: 0.1422
